# Clase 145 — Hugging Face Transformers (uso práctico)

La librería **`transformers`** es el estándar de la industria para usar modelos
preentrenados (BERT, GPT, T5, Llama, Whisper, ViT...). Recorremos la `pipeline`
API (one-liner), los componentes manuales (`AutoTokenizer` + `AutoModel*`) y el
`Trainer` para fine-tuning.

**Requiere:** `transformers`, `torch` (y descarga de modelos desde el Hub). Si no
están instalados, cada celda avisa y no ejecuta la parte que depende de la red;
el código mostrado es la **API real y vigente** de Hugging Face.

## 1. Entorno: detectar `transformers`

In [ ]:
try:
    import transformers
    from transformers import pipeline
    HAS_TF = True
    print('transformers:', transformers.__version__)
except Exception as e:
    HAS_TF = False
    print('transformers no instalado. No se descargan modelos. Motivo:', type(e).__name__)

import numpy as np
np.random.seed(42)

## 2. `pipeline`: el one-liner que resuelve el 90 % de los casos

`pipeline(task)` descarga un modelo por defecto para la tarea y devuelve un
callable. Cubre `sentiment-analysis`, `ner`, `question-answering`, `fill-mask`,
`summarization`, `zero-shot-classification`, `translation`, etc.

In [ ]:
if HAS_TF:
    sentiment = pipeline('sentiment-analysis')
    print(sentiment('I absolutely loved this movie!'))
    # -> [{'label': 'POSITIVE', 'score': 0.999...}]

    fill = pipeline('fill-mask', model='bert-base-uncased')
    print(fill('Paris is the [MASK] of France.')[:2])
else:
    print('Salida esperada de sentiment-analysis:')
    print("[{'label': 'POSITIVE', 'score': 0.9998}]")

## 3. Más tareas: zero-shot, NER y question-answering

`zero-shot-classification` clasifica contra etiquetas arbitrarias sin haber sido
entrenado en ellas (usa NLI por debajo).

In [ ]:
if HAS_TF:
    zshot = pipeline('zero-shot-classification')
    r = zshot('The new GPU doubles the training throughput.',
              candidate_labels=['sports', 'politics', 'tech'])
    print(dict(zip(r['labels'], np.round(r['scores'], 3))))

    ner = pipeline('ner', aggregation_strategy='simple')
    print(ner('Hugging Face was founded in New York.'))

    qa = pipeline('question-answering')
    print(qa(question='Where was it founded?',
             context='Hugging Face was founded in New York.'))
else:
    print('zero-shot -> tech domina; ner -> ORG/LOC; qa -> answer="New York"')

## 4. API manual: `AutoTokenizer` + `AutoModel*`

Para custom loops se cargan por separado el tokenizer y el modelo con la cabeza
adecuada (`AutoModelForSequenceClassification`, `...ForCausalLM`, etc.). El
tokenizer hace **subword tokenization** (WordPiece en BERT): sin OOV, vocab fijo.

In [ ]:
if HAS_TF:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained('bert-base-uncased')
    enc = tok(['Transformers are powerful.', 'I dislike bugs.'],
              padding=True, truncation=True, max_length=32, return_tensors='pt')
    print('input_ids shape:', enc['input_ids'].shape)
    print('keys:', list(enc.keys()))          # input_ids, token_type_ids, attention_mask
    print('decode:', tok.decode([101, 7592, 102]))  # '[CLS] hello [SEP]'
else:
    print('input_ids, token_type_ids y attention_mask; [CLS]=101, [SEP]=102')

## 5. Forward pass del modelo → `logits`

In [ ]:
if HAS_TF:
    import torch
    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased-finetuned-sst-2-english')
    tok2 = AutoTokenizer.from_pretrained(
        'distilbert-base-uncased-finetuned-sst-2-english')
    inputs = tok2('This library is fantastic!', return_tensors='pt')
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)
    print('logits:', outputs.logits)
    print('probs :', probs, '| labels:', model.config.id2label)
else:
    # softmax es puro numpy: ilustramos el paso logits -> probabilidades
    def softmax(z):
        z = z - z.max()
        e = np.exp(z)
        return e / e.sum()
    print('softmax([-2.1, 3.4]) =', np.round(softmax(np.array([-2.1, 3.4])), 3))

## 6. Fine-tuning con `Trainer` + `TrainingArguments`

Patrón estándar (95 % de los casos): cargar dataset con `datasets`, tokenizar,
`AutoModelForSequenceClassification(num_labels=...)`, definir `TrainingArguments`
y `Trainer(...).train()`. LR típico para fine-tuning: `2e-5`.

In [ ]:
if HAS_TF:
    from datasets import load_dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)

    ds = load_dataset('imdb')
    tok = AutoTokenizer.from_pretrained('distilbert-base-uncased')

    def tokenize(batch):
        return tok(batch['text'], truncation=True, padding='max_length', max_length=256)

    ds_tok = ds.map(tokenize, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=2)

    args = TrainingArguments(output_dir='out', num_train_epochs=2,
                             per_device_train_batch_size=16,
                             learning_rate=2e-5, eval_strategy='epoch')
    trainer = Trainer(model=model, args=args,
                      train_dataset=ds_tok['train'], eval_dataset=ds_tok['test'])
    # trainer.train()   # descomentar con GPU; DistilBERT ~0.93 acc en IMDB
    print('Trainer configurado (train() comentado: requiere GPU y descarga).')
else:
    print('Trainer(model, args, train_dataset, eval_dataset).train() -> fine-tuning en ~20 lineas.')

## Ejercicios

1. **pipeline one-liner**: probá `pipeline('sentiment-analysis')` con 3 frases
   (positiva, negativa, ambigua) e inspeccioná `label` y `score`.
2. **Zero-shot**: clasificá un titular con `candidate_labels=['deportes','política','tecnología']`.
3. **Tokenización manual**: tokenizá una frase con `bert-base-uncased` e imprimí
   `input_ids`, `attention_mask` y `token_type_ids`; luego `tokenizer.decode(...)`.
4. **Fine-tuning**: adaptá la celda 6 a `distilbert` sobre IMDB (2 épocas, `lr=2e-5`)
   y reportá accuracy en test (criterio del README: ≥ 0.92).

## Conclusiones

- `pipeline(task)` resuelve inferencia en una línea para decenas de tareas.
- `AutoTokenizer` + `AutoModelFor<Task>` dan control total para custom loops;
  el forward devuelve `logits` que se pasan por `softmax`.
- La subword tokenization (WordPiece/BPE/SentencePiece) elimina el OOV con vocab fijo.
- `Trainer` + `TrainingArguments` estandarizan el fine-tuning; `learning_rate=2e-5`
  evita catastrophic forgetting en modelos grandes.
- El Hub (huggingface.co/models) es el catálogo de +500k modelos preentrenados.